# CartPole DQN
**Algorithm:** Deep Q-Network  
**Training target:** 5,000 completed episodes  
**Checkpoint:** every 1,000 completed episodes

## 1. Setup
Install Gymnasium, Stable-Baselines3, and plotting/video packages.

In [ ]:
%pip install -q \
    "gymnasium[classic-control]>=1.0" \
    "stable-baselines3[extra]>=2.4" \
    "imageio[ffmpeg]" \
    "pandas" \
    "matplotlib"

## 2. Configuration
Set the seed, folders, and device.

In [ ]:
import re
import time
import random
import base64
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio

import gymnasium as gym

import torch as th
import torch.nn as nn

from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.utils import set_random_seed
from IPython import display as ipythondisplay

SEED = 42
EVAL_EPISODES = 100
CHUNK_STEPS = 25_000
LOG_INTERVAL_EPISODES = 50

random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
set_random_seed(SEED)

DEVICE = (
    "cuda"
    if th.cuda.is_available()
    else "cpu"
)

ROOT = Path("as2_cartpole")
MODEL_DIR = ROOT / "models"
CHECKPOINT_DIR = ROOT / "checkpoints"
LOG_DIR = ROOT / "logs"
VIDEO_DIR = ROOT / "videos"

for folder in [
    MODEL_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    VIDEO_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Device:", DEVICE)


from stable_baselines3 import DQN

## 3. Fixed Hyperparameters
Training target, checkpoint interval, DQN settings, and ANN settings are separated from the training loop.

In [ ]:
TOTAL_EPISODES = 5_000
CHECKPOINT_INTERVAL_EPISODES = 1_000
LOG_INTERVAL_EPISODES = 50

DQN_LEARNING_RATE = 1e-3
DQN_BUFFER_SIZE = 150_000
DQN_LEARNING_STARTS = 1_000
DQN_BATCH_SIZE = 64
DQN_GAMMA = 0.99
DQN_TRAIN_FREQ = 4
DQN_GRADIENT_STEPS = 1
DQN_TARGET_UPDATE_INTERVAL = 500
DQN_EXPLORATION_FRACTION = 0.20
DQN_EXPLORATION_INITIAL_EPS = 1.0
DQN_EXPLORATION_FINAL_EPS = 0.02

ANN_FEATURES_DIM = 128

## 4. Problem Formulation and Custom ANN

**Environment:** `CartPole-v1`

**State space:** four continuous values representing cart position, cart velocity, pole angle, and pole angular velocity.

**Action space:** two discrete actions: move the cart left or move the cart right.

**Reward:** +1 for every timestep that the pole remains balanced. An episode ends when the pole or cart exceeds the environment limits, or when the time limit is reached.

**Goal:** maximise cumulative episode reward by keeping the pole balanced for as long as possible.

A custom ANN is used as the Stable-Baselines3 feature extractor.

In [ ]:
ENV_ID = "CartPole-v1"


def make_env(
    seed=SEED,
    render_mode=None,
):
    def _init():
        env = gym.make(
            ENV_ID,
            render_mode=render_mode,
        )

        env = Monitor(env)
        env.reset(seed=seed)
        env.action_space.seed(seed)
        return env

    return _init


class CustomANN(BaseFeaturesExtractor):
    def __init__(
        self,
        observation_space,
        features_dim=ANN_FEATURES_DIM,
    ):
        super().__init__(
            observation_space,
            features_dim,
        )

        input_dim = int(
            np.prod(
                observation_space.shape
            )
        )

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                input_dim,
                128,
            ),
            nn.ReLU(),
            nn.Linear(
                128,
                features_dim,
            ),
            nn.ReLU(),
        )

    def forward(
        self,
        observations,
    ):
        return self.net(
            observations
        )


POLICY_KWARGS = {
    "features_extractor_class": CustomANN,
    "features_extractor_kwargs": {
        "features_dim": ANN_FEATURES_DIM,
    },
}

## 5. Utilities
Utilities handle episode logging, checkpointing, evaluation, plotting, and video output.

In [ ]:
class EpisodeTrainingCallback(BaseCallback):
    def __init__(
        self,
        log_path,
        checkpoint_dir,
        checkpoint_prefix,
        target_episodes,
        start_episode=0,
    ):
        super().__init__()

        self.log_path = Path(
            log_path
        )

        self.checkpoint_dir = Path(
            checkpoint_dir
        )

        self.checkpoint_prefix = (
            checkpoint_prefix
        )

        self.target_episodes = int(
            target_episodes
        )

        self.completed_episodes = int(
            start_episode
        )

        self.next_checkpoint = (
            (
                self.completed_episodes
                // CHECKPOINT_INTERVAL_EPISODES
            )
            + 1
        ) * CHECKPOINT_INTERVAL_EPISODES

        self.next_log = (
            (
                self.completed_episodes
                // LOG_INTERVAL_EPISODES
            )
            + 1
        ) * LOG_INTERVAL_EPISODES

        self.log_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.checkpoint_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

    def _append_row(
        self,
        row,
    ):
        new_row = pd.DataFrame(
            [row]
        )

        if self.log_path.exists():
            existing = pd.read_csv(
                self.log_path
            )

            new_row = pd.concat(
                [
                    existing,
                    new_row,
                ],
                ignore_index=True,
            )

            new_row = (
                new_row
                .drop_duplicates(
                    subset=["episode"],
                    keep="last",
                )
                .sort_values(
                    "episode"
                )
            )

        new_row.to_csv(
            self.log_path,
            index=False,
        )

    def _on_step(self):
        for info in self.locals.get(
            "infos",
            [],
        ):
            if "episode" not in info:
                continue

            self.completed_episodes += 1

            row = {
                "episode": self.completed_episodes,
                "timestep": int(
                    self.model.num_timesteps
                ),
                "reward": float(
                    info["episode"]["r"]
                ),
                "length": int(
                    info["episode"]["l"]
                ),
            }

            self._append_row(row)

            if (
                self.completed_episodes
                >= self.next_log
            ):
                recent = pd.read_csv(
                    self.log_path
                ).tail(
                    LOG_INTERVAL_EPISODES
                )

                print(
                    f"episode="
                    f"{self.completed_episodes:,} | "
                    f"mean_reward="
                    f"{recent['reward'].mean():.2f} | "
                    f"mean_length="
                    f"{recent['length'].mean():.1f}"
                )

                while (
                    self.next_log
                    <= self.completed_episodes
                ):
                    self.next_log += (
                        LOG_INTERVAL_EPISODES
                    )

            if (
                self.completed_episodes
                >= self.next_checkpoint
            ):
                path = (
                    self.checkpoint_dir
                    / (
                        f"{self.checkpoint_prefix}_"
                        f"{self.completed_episodes}_episodes"
                    )
                )

                self.model.save(
                    path
                )

                print(
                    "Checkpoint saved:",
                    str(path) + ".zip",
                )

                while (
                    self.next_checkpoint
                    <= self.completed_episodes
                ):
                    self.next_checkpoint += (
                        CHECKPOINT_INTERVAL_EPISODES
                    )

            if (
                self.completed_episodes
                >= self.target_episodes
            ):
                return False

        return True


def latest_episode_checkpoint(
    folder,
    prefix,
):
    files = list(
        Path(folder).glob(
            f"{prefix}_*_episodes.zip"
        )
    )

    if not files:
        return None, 0

    def get_episode(path):
        match = re.search(
            r"_(\d+)_episodes\.zip$",
            path.name,
        )

        return (
            int(match.group(1))
            if match
            else -1
        )

    latest = max(
        files,
        key=get_episode,
    )

    return (
        latest,
        get_episode(latest),
    )


def load_episode_log(
    path,
):
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    return (
        pd.read_csv(path)
        .drop_duplicates(
            subset=["episode"],
            keep="last",
        )
        .sort_values(
            "episode"
        )
        .reset_index(
            drop=True
        )
    )


def plot_episode_curve(
    path,
    title,
):
    frame = load_episode_log(
        path
    )

    if frame.empty:
        print(
            "No saved training log found."
        )
        return frame

    frame["mean_100"] = (
        frame["reward"]
        .rolling(
            100,
            min_periods=1,
        )
        .mean()
    )

    plt.figure(
        figsize=(9, 4)
    )

    plt.plot(
        frame["episode"],
        frame["reward"],
        alpha=0.25,
        label="Episode reward",
    )

    plt.plot(
        frame["episode"],
        frame["mean_100"],
        linewidth=2,
        label="100-episode mean",
    )

    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

    return frame


def evaluate_model(
    model,
    episodes=EVAL_EPISODES,
):
    env = DummyVecEnv([
        make_env(
            SEED + 1000
        )
    ])

    rewards, lengths = evaluate_policy(
        model,
        env,
        n_eval_episodes=episodes,
        deterministic=True,
        return_episode_rewards=True,
        warn=False,
    )

    env.close()

    return {
        "mean_reward": float(
            np.mean(rewards)
        ),
        "std_reward": float(
            np.std(rewards)
        ),
        "median_reward": float(
            np.median(rewards)
        ),
        "mean_length": float(
            np.mean(lengths)
        ),
        "max_reward": float(
            np.max(rewards)
        ),
    }


def evaluate_all_episode_checkpoints(
    model_class,
    folder,
    prefix,
):
    files = list(
        Path(folder).glob(
            f"{prefix}_*_episodes.zip"
        )
    )

    if not files:
        return pd.DataFrame()

    def get_episode(path):
        match = re.search(
            r"_(\d+)_episodes\.zip$",
            path.name,
        )
        return int(
            match.group(1)
        )

    rows = []

    for path in sorted(
        files,
        key=get_episode,
    ):
        episode = get_episode(
            path
        )

        print(
            f"Evaluating "
            f"{episode:,} episodes..."
        )

        model = model_class.load(
            path,
            device=DEVICE,
        )

        result = evaluate_model(
            model
        )

        rows.append({
            "episode": episode,
            **result,
        })

        del model
        gc.collect()

        if th.cuda.is_available():
            th.cuda.empty_cache()

    return pd.DataFrame(
        rows
    )


def random_baseline():
    rewards = []
    lengths = []

    for episode in range(
        EVAL_EPISODES
    ):
        env = gym.make(
            ENV_ID
        )

        episode_seed = SEED + episode

        obs, _ = env.reset(
            seed=episode_seed
        )

        env.action_space.seed(
            episode_seed
        )

        total_reward = 0.0
        total_length = 0
        terminated = False
        truncated = False

        while not (
            terminated
            or truncated
        ):
            obs, reward, terminated, truncated, _ = (
                env.step(
                    env.action_space.sample()
                )
            )

            total_reward += float(
                reward
            )
            total_length += 1

        rewards.append(
            total_reward
        )
        lengths.append(
            total_length
        )

        env.close()

    return {
        "mean_reward": float(
            np.mean(rewards)
        ),
        "std_reward": float(
            np.std(rewards)
        ),
        "median_reward": float(
            np.median(rewards)
        ),
        "mean_length": float(
            np.mean(lengths)
        ),
        "max_reward": float(
            np.max(rewards)
        ),
    }


def show_video(
    path,
):
    path = Path(path)

    video_b64 = (
        base64.b64encode(
            path.read_bytes()
        )
        .decode("ascii")
    )

    ipythondisplay.display(
        ipythondisplay.HTML(
            f"""
            <video controls
                   style="height: 400px;
                          max-width: 100%;">
                <source
                    src="data:video/mp4;base64,{video_b64}"
                    type="video/mp4">
            </video>
            """
        )
    )


def record_model_video(
    model,
    filename,
    seed=SEED + 5000,
):
    env = gym.make(
        ENV_ID,
        render_mode="rgb_array",
    )

    obs, _ = env.reset(
        seed=seed
    )

    frames = [
        env.render()
    ]

    terminated = False
    truncated = False

    while not (
        terminated
        or truncated
    ):
        action, _ = model.predict(
            obs,
            deterministic=True,
        )

        obs, reward, terminated, truncated, info = (
            env.step(
                int(
                    np.asarray(action).item()
                )
            )
        )

        frames.append(
            env.render()
        )

    env.close()

    path = VIDEO_DIR / filename

    imageio.mimsave(
        path,
        frames,
        fps=30,
    )

    return path

## 6. Random Baseline
Evaluate random actions before training.

In [ ]:
random_result = (
    random_baseline()
)

random_result

## 7. DQN Model Initialization and Training

The model resumes from the latest saved checkpoint when one is available. Training continues until the fixed episode target is reached.

In [ ]:
DQN_CHECKPOINT_DIR = (
    CHECKPOINT_DIR
    / "dqn"
)

DQN_LOG_PATH = (
    LOG_DIR
    / "dqn_training_log.csv"
)

DQN_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dqn_env = DummyVecEnv([
    make_env(SEED)
])

latest_checkpoint, completed_episodes = (
    latest_episode_checkpoint(
        DQN_CHECKPOINT_DIR,
        "cartpole_dqn",
    )
)

if latest_checkpoint is None:
    dqn = DQN(
        "MlpPolicy",
        dqn_env,
        learning_rate=DQN_LEARNING_RATE,
        buffer_size=DQN_BUFFER_SIZE,
        learning_starts=DQN_LEARNING_STARTS,
        batch_size=DQN_BATCH_SIZE,
        gamma=DQN_GAMMA,
        train_freq=DQN_TRAIN_FREQ,
        gradient_steps=DQN_GRADIENT_STEPS,
        target_update_interval=DQN_TARGET_UPDATE_INTERVAL,
        exploration_fraction=DQN_EXPLORATION_FRACTION,
        exploration_initial_eps=DQN_EXPLORATION_INITIAL_EPS,
        exploration_final_eps=DQN_EXPLORATION_FINAL_EPS,
        policy_kwargs=POLICY_KWARGS,
        verbose=0,
        seed=SEED,
        device=DEVICE,
    )

    completed_episodes = 0

else:
    print(
        "Resuming from:",
        latest_checkpoint,
    )

    dqn = DQN.load(
        latest_checkpoint,
        env=dqn_env,
        device=DEVICE,
    )

training_callback = EpisodeTrainingCallback(
    DQN_LOG_PATH,
    DQN_CHECKPOINT_DIR,
    "cartpole_dqn",
    TOTAL_EPISODES,
    start_episode=completed_episodes,
)

start = time.time()

while (
    training_callback.completed_episodes
    < TOTAL_EPISODES
):
    dqn.learn(
        total_timesteps=CHUNK_STEPS,
        reset_num_timesteps=False,
        callback=training_callback,
        progress_bar=False,
    )

print(
    "Training time:",
    round(
        (
            time.time()
            - start
        )
        / 60,
        1,
    ),
    "minutes",
)

dqn.save(
    MODEL_DIR
    / "cartpole_dqn_final"
)

## 8. Learning Curve

Training progress is loaded from the persistent episode CSV log. The 100-episode moving mean is used to show the learning trend.

In [ ]:
dqn_curve = plot_episode_curve(
    DQN_LOG_PATH,
    "CartPole DQN Learning Curve",
)

display(
    dqn_curve.tail(10)
)

## 9. Performance Evaluation

Saved checkpoints are evaluated using deterministic actions. Mean reward, standard deviation, median reward, mean episode length, and maximum reward are reported.

In [ ]:
dqn_results = (
    evaluate_all_episode_checkpoints(
        DQN,
        DQN_CHECKPOINT_DIR,
        "cartpole_dqn",
    )
)

display(
    dqn_results
)

best_dqn = (
    dqn_results
    .sort_values(
        "mean_reward",
        ascending=False,
    )
    .iloc[0]
)

best_dqn

## 10. Results
Compare the random baseline with the best DQN checkpoint.

In [ ]:
cartpole_dqn_results = (
    pd.DataFrame([
        {
            "Algorithm": "Random",
            **random_result,
        },
        {
            "Algorithm": "DQN",
            "mean_reward": best_dqn[
                "mean_reward"
            ],
            "std_reward": best_dqn[
                "std_reward"
            ],
            "median_reward": best_dqn[
                "median_reward"
            ],
            "mean_length": best_dqn[
                "mean_length"
            ],
            "max_reward": best_dqn[
                "max_reward"
            ],
        },
    ])
)

cartpole_dqn_results

## 11. Simulation Video

The best DQN checkpoint is simulated, recorded, and displayed directly in the notebook.

In [ ]:
best_episode = int(
    best_dqn["episode"]
)

best_checkpoint = (
    DQN_CHECKPOINT_DIR
    / (
        "cartpole_dqn_"
        f"{best_episode}_episodes.zip"
    )
)

video_model = DQN.load(
    best_checkpoint,
    device=DEVICE,
)

video_path = (
    record_model_video(
        video_model,
        "cartpole-dqn-best.mp4",
    )
)

show_video(
    video_path
)

del video_model
gc.collect()

if th.cuda.is_available():
    th.cuda.empty_cache()